# 02 — U.S. Census counties + browser GeoAI clustering

**[🚀 Launch this notebook live](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/lite/lab/index.html?path=v2_02_census_geoai_counties.ipynb)**

This notebook uses the Census Bureau's **TIGERweb** GeoJSON service rather than the Census Data API, so it can run without an API key. We create county-level spatial features for Illinois and use scikit-learn K-Means as a lightweight GeoAI pattern.

The model is exploratory: clusters are not policy categories and should not be interpreted as causal or normative labels.

> **JupyterLite compatibility:** this notebook uses `geolibre_lite.LiteMap`, which builds native GeoLibre project/layer definitions and displays them through GeoLibre's hosted `?embed=1` viewer. It deliberately avoids `geolibre.Map`, whose localhost HTTP server cannot bind a socket inside Pyodide/WebAssembly.


In [ ]:
import sys
if sys.platform == "emscripten":
    import micropip
    await micropip.install(["geolibre==3.0.0", "pyodide-http"])
from geolibre_lite import LiteMap as Map


In [ ]:
import sys, requests, pandas as pd, numpy as np
if sys.platform == "emscripten":
    import pyodide_http
    pyodide_http.patch_all()

import geopandas as gpd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans


In [ ]:
url = "https://tigerweb.geo.census.gov/arcgis/rest/services/TIGERweb/State_County/MapServer/57/query"
params = {
    "where": "STATE='17'",
    "outFields": "GEOID,BASENAME,POP100,HU100,AREALAND,AREAWATER",
    "returnGeometry": "true",
    "outSR": "4326",
    "f": "geojson",
}
response = requests.get(url, params=params, timeout=60)
response.raise_for_status()
data = response.json()
gdf = gpd.GeoDataFrame.from_features(data["features"], crs="EPSG:4326")
gdf.head()


In [ ]:
# Browser-side feature engineering.
metric = gdf.to_crs(5070)
gdf["area_km2"] = metric.area / 1e6
gdf["population_density"] = gdf["POP100"].astype(float) / gdf["area_km2"]
gdf["housing_density"] = gdf["HU100"].astype(float) / gdf["area_km2"]
gdf["people_per_housing_unit"] = (
    gdf["POP100"].astype(float) / gdf["HU100"].astype(float).replace(0, np.nan)
).fillna(0)

feature_cols = ["population_density", "housing_density", "people_per_housing_unit"]
X = np.log1p(gdf[feature_cols].clip(lower=0))
Xz = StandardScaler().fit_transform(X)
gdf["cluster"] = KMeans(n_clusters=5, random_state=42, n_init=20).fit_predict(Xz)
gdf[["BASENAME", *feature_cols, "cluster"]].sort_values("population_density", ascending=False).head(10)


In [ ]:
m = Map(center=(-89.2, 40.0), zoom=5, height="720px")
m.add_choropleth(
    gdf,
    column="population_density",
    name="2020 population density",
    class_count=7,
    colormap="viridis",
    scheme="quantile",
)
m.add_choropleth(
    gdf,
    column="cluster",
    name="K-Means feature cluster",
    class_count=5,
    colormap="turbo",
    scheme="equal-interval",
)
m


## Why this is GeoAI-relevant

GeoAI is broader than deep neural networks. A common geospatial AI pipeline is:

1. obtain geographic features;
2. engineer spatially meaningful predictors;
3. standardize/transform them;
4. learn a pattern or class;
5. write model outputs back to geometry;
6. inspect the result on an interactive map.

The full `geoai-py` package supports much richer imagery/deep-learning workflows; this browser lab concentrates on the part that can run reliably in WebAssembly.

## Data & software citations

- U.S. Census Bureau TIGERweb State/County MapServer: https://tigerweb.geo.census.gov/arcgis/rest/services/TIGERweb/State_County/MapServer
- Geometry/attributes are from the Census 2020 county layer exposed by TIGERweb. Source/copyright: U.S. Census Bureau.
- scikit-learn K-Means: https://scikit-learn.org/
- GeoLibre: https://geolibre.app/.

For research use, verify the exact TIGERweb vintage and preserve the query URL/parameters with your output.
